In [0]:
%sql
-- Create Delta tables for storing movie text data and tracking processed CSV files
-- rag_docs_text: Stores movie text (title + genre + overview) for vector search embedding
-- rag_docs_track: Tracks which CSV files have been processed to avoid duplicates
-- Both tables enable Change Data Feed for vector search sync

CREATE OR REPLACE TABLE rag_capstone_catalog.default.rag_docs_text (
  id BIGINT GENERATED BY DEFAULT AS IDENTITY NOT NULL,
  title STRING,
  genre STRING,
  text STRING,
  adult BOOLEAN,
  genres STRING,
  languages STRING,
  file_name STRING,
  CONSTRAINT rag_docs_text_pk PRIMARY KEY (id)
) TBLPROPERTIES (delta.enableChangeDataFeed = true);


CREATE OR REPLACE TABLE rag_capstone_catalog.default.rag_docs_track (
  id BIGINT GENERATED BY DEFAULT AS IDENTITY,
  file_name STRING
) TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
# Install libraries for CSV processing and text chunking
# - langchain-text-splitters: Text splitting utilities for chunking movie descriptions
# - pandas: CSV file reading (pre-installed on Databricks but listed for clarity)

%pip install langchain-text-splitters pandas

dbutils.library.restartPython()

In [0]:
# List all CSV files in the documents volume and extract their file names
import os
from pyspark.sql.functions import substring_index
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()


# Directory containing CSV files to process
directory_path = "/Volumes/rag_capstone_catalog/default/documents"

# List files in directory (filter for CSV files only)
file_paths = [file.path for file in dbutils.fs.ls(directory_path) if file.path.endswith(".csv")]

# Extract file names from paths
df = spark.createDataFrame(file_paths, "string").select(substring_index("value", "/", -1).alias("file_name"))
                                                                        
#Show dataframe
df.show()


## CSV Processing Check
Identify which CSV files are new and haven't been processed yet

In [0]:
# Read new CSV files and extract movie text for vector search
# Only processes files that haven't been tracked in rag_docs_track table
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter

csv_volume_path = "/Volumes/rag_capstone_catalog/default/documents"

# Get list of already processed CSV files from the Delta Table
try:
    processed_files = spark.sql("SELECT DISTINCT file_name FROM rag_capstone_catalog.default.rag_docs_track").collect()
    processed_files = set(row["file_name"] for row in processed_files)
except:
    # Table doesn't exist yet (first run) - treat all files as new
    processed_files = set()

# List CSV files in the volume directory
all_csv_files = [file.name for file in dbutils.fs.ls(csv_volume_path) if file.name.endswith(".csv")]

# Process only new CSV files
new_files = [file for file in all_csv_files if file not in processed_files]

# Initialize list to store movie records
# (title, genre, text, adult, genres, languages, file_name)
movie_records = []

# Initialize text splitter for long descriptions
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

for file_name in new_files:
    csv_path = f"{csv_volume_path}/{file_name}"
    # Read CSV with pandas
    csv_df = pd.read_csv(csv_path)
    
    for _, row in csv_df.iterrows():
        title = str(row.get("title", row.get("Title", "")))
        genre = str(row.get("genre", row.get("Genre", "")))
        genres = str(row.get("genres", row.get("Genres", "")))
        adult = row.get("adult", row.get("Adult", False))
        # Handle boolean conversion from CSV string
        if isinstance(adult, str):
            adult = adult.lower().strip() in ("true", "1", "yes")
        languages = str(row.get("languages", row.get("Languages", row.get("spoken_languages", ""))))
        overview = str(row.get("overview", row.get("description", row.get("Description", row.get("synopsis", "")))))
        
        # Combine all movie fields into a single text for embedding
        combined_text = f"Title: {title}. Genre: {genre}. Genres: {genres}. Adult: {adult}. Languages: {languages}. Overview: {overview}"
        
        # Chunk the combined text if it's long
        chunks = text_splitter.split_text(combined_text)
        
        for chunk in chunks:
            movie_records.append((title, genre, chunk, adult, genres, languages, file_name))

# Display sample records
print(f"Total movie records to insert: {len(movie_records)}")
if movie_records:
    print("Sample:", movie_records[0])

## Pandas UDF for Chunk Processing
Define UDF to efficiently handle movie text chunks for batch insertion

In [0]:
# Define and register Pandas UDF to return movie text chunks as an array
# This enables efficient batch processing of movie data in Spark
#from pyspark.sql.functions import pandas_udf
#from pyspark.sql.types import ArrayType, StringType
#import pandas as pd

#@pandas_udf("array<string>")
#def get_chunks(dummy):
 #   return pd.Series(chunks)

# Register the UDF for SQL usage
#spark.udf.register("get_chunks_udf", get_chunks)


In [0]:
# Insert movie text records into rag_docs_text table for vector search indexing
# Only inserts if new records were extracted from CSVs
if movie_records:
    records_df = spark.createDataFrame(movie_records, ["title", "genre", "text", "adult", "genres", "languages", "file_name"])
    records_df.write.mode("append").saveAsTable("rag_capstone_catalog.default.rag_docs_text")
    records_df.show()
else:
    print("No new records to process - all files have already been processed.")

In [0]:
# Track newly processed files in rag_docs_track table to prevent reprocessing
# Uses temporary view to check for duplicates before insertion
df.createOrReplaceTempView("temp_table")

# Insert only file names that don't already exist in the target table

spark.sql("""
          INSERT INTO rag_capstone_catalog.default.rag_docs_track (file_name)
          SELECT file_name FROM temp_table
          WHERE NOT EXISTS (
              SELECT 1 FROM rag_capstone_catalog.default.rag_docs_track
              WHERE temp_table.file_name = rag_capstone_catalog.default.rag_docs_track.file_name
          )
          
          """)


In [0]:
# Create Vector Search endpoint and Delta Sync index for movie embeddings
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    EndpointType, VectorIndexType, PipelineType,
    DeltaSyncVectorIndexSpecRequest, EmbeddingSourceColumn
)

w = WorkspaceClient()

# Create a Vector Search endpoint (STANDARD for low-latency retrieval)
endpoint_name = "capstoneendpoint4"

try:
    w.vector_search_endpoints.create_endpoint(
        name=endpoint_name,
        endpoint_type=EndpointType.STANDARD
    )
    print(f"Creating endpoint '{endpoint_name}'...")
except Exception as e:
    print(f"Endpoint '{endpoint_name}' may already exist: {e}")

# Create Delta Sync index with managed embeddings on the text column
index_name = "rag_capstone_catalog.default.rag_docs_text_index4"

try:
    w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name=endpoint_name,
        primary_key="id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table="rag_capstone_catalog.default.rag_docs_text",
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name="text",
                    embedding_model_endpoint_name="databricks-gte-large-en"
                )
            ],
            columns_to_sync=[
                "id", "title", "genre", "text",
                "adult", "genres", "languages", "file_name"
            ],
            pipeline_type=PipelineType.TRIGGERED
        )
    )
    print(f"Creating index '{index_name}'...")
except Exception as e:
    print(f"Index '{index_name}' may already exist: {e}")

In [0]:
%sql
SELECT * FROM rag_capstone_catalog.default.movie_conversation_history
  WHERE session_id = 'test_user_123Capstone'